In [1]:
import torch

from transformers import AutoTokenizer

#加载tokenizer
tokenizer = AutoTokenizer.from_pretrained(r'C:\Users\ssw\Desktop\CS_Code\Machine_learn\ner\model\bert-base-chinese')

tokenizer

C:\Users\ssw\.conda\envs\ner\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BertTokenizerFast(name_or_path='C:\Users\ssw\Desktop\CS_Code\Machine_learn\ner\model\bert-base-chinese', vocab_size=21128, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [2]:
from datasets import load_dataset

#加载数据集
dataset = load_dataset(path='lansinuote/ChnSentiCorp')

#编码
f = lambda x: tokenizer(
    x['text'], truncation=True, max_length=30, return_token_type_ids=False)
dataset = dataset.map(f, remove_columns=['text', 'label'])

#过滤句子长度
f = lambda x: len(x['input_ids']) >= 30
dataset = dataset.filter(f)


#重置label字段
def f(data):
    #定义第15个字为label
    data['label'] = data['input_ids'][15]

    #替换句子中的第15个字为mask
    data['input_ids'][15] = tokenizer.mask_token_id

    return data


dataset = dataset.map(f)

#设置数据类型 pytorch tensor 格式
dataset.set_format('pt')

dataset, dataset['train'][0]

Map: 100%|██████████| 1157/1157 [00:00<00:00, 19337.99 examples/s]


(DatasetDict({
     train: Dataset({
         features: ['input_ids', 'attention_mask', 'label'],
         num_rows: 9286
     })
     validation: Dataset({
         features: ['input_ids', 'attention_mask', 'label'],
         num_rows: 1158
     })
     test: Dataset({
         features: ['input_ids', 'attention_mask', 'label'],
         num_rows: 1157
     })
 }),
 {'input_ids': tensor([ 101, 6848, 2885, 4403, 3736, 5709, 1736, 4638, 1333, 1728, 2218, 3221,
          3175,  912, 8024,  103, 4510, 1220, 2820, 3461, 4684, 2970, 1168, 6809,
          3862, 6804, 8024, 1453, 1741,  102]),
  'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1]),
  'label': tensor(3300)})

In [3]:
loader = torch.utils.data.DataLoader(dataset['train'],
                                     batch_size=8,
                                     shuffle=True,
                                     drop_last=True)

data = next(iter(loader))

for k, v in data.items():
    print(k, v.shape)

len(loader)

input_ids torch.Size([8, 30])
attention_mask torch.Size([8, 30])
label torch.Size([8])


1160

In [4]:
#查看数据样例,将中间词替换为了[MASK], 接下来的任务就是预测这个掩码
for q, a in zip(data['input_ids'], data['label']):
    print(tokenizer.decode(q))
    print(tokenizer.decode(a))
    print('==============')

[CLS] 10 月 24 日 入 住 西 楼, 房 间 很 宽 敞 [MASK] 但 设 施 旧 了 一 些, 它 的 房 间 有 [SEP]
,
[CLS] 在 太 原 还 算 不 错 的 ， 个 人 认 为 要 [MASK] 山 西 大 酒 店 好 ！ 国 贸 没 住 过 ， [SEP]
比
[CLS] 位 置 太 难 找 了 ， 房 间 设 施 少 ， 功 [MASK] 单 一 ， 作 为 商 务 间 ， 没 有 配 备 [SEP]
能
[CLS] 比 朋 友 的 ideapad [UNK] 大 了 一 圈 ， 不 过 [MASK] 虑 键 盘 比 较 大 而 且 电 池 时 间 长 [SEP]
考
[CLS] 不 能 住 靠 马 路 的 房 间, 太 吵, 一 [MASK] 上 没 睡 好. 旁 边 有 个 大 超 市, [SEP]
晚
[CLS] 隔 音 效 果 非 常 非 常 非 常 的 差 ， 睡 [MASK] 房 间 里 象 睡 在 马 路 边 一 样 ， 路 [SEP]
在
[CLS] 从 拿 到 书 开 始 我 就 已 经 知 道 我 家 [MASK] 宝 贝 一 定 会 喜 欢 的 ， 从 文 字 和 [SEP]
的
[CLS] 太 多 了 不 一 一 细 说 主 要 是 价 格 合 [MASK] [UNK] 感 觉 比 我 以 前 用 的 扣 肉 2 1 [SEP]
理


In [6]:
#定义模型
class Model(torch.nn.Module):

    def __init__(self):
        super().__init__()

        #加载预训练模型
        from transformers import AutoModel
        self.pretrained = AutoModel.from_pretrained(
            r'C:\Users\ssw\Desktop\CS_Code\Machine_learn\ner\model\bert-base-chinese')

        self.fc = torch.nn.Linear(in_features=768,
                                  out_features=tokenizer.vocab_size)

    def forward(self, input_ids, attention_mask, label=None):
        #使用预训练模型抽取数据特征
        with torch.no_grad():
            last_hidden_state = self.pretrained(
                input_ids=input_ids,
                attention_mask=attention_mask).last_hidden_state

        #取第15个词的特征向量
        last_hidden_state = last_hidden_state[:, 15]

        #对抽取的特征只取第一个字的结果做分类即可
        #所以简单来说,其实这个任务本质也是做分类任务(只不过词特别多有15个)
        out = self.fc(last_hidden_state).softmax(dim=1)

        #计算loss(与情感分类一致)
        loss = None
        if label is not None:
            loss = torch.nn.functional.cross_entropy(out, label)

        return loss, out


model = Model()

model(**data)

(tensor(9.9584, grad_fn=<NllLossBackward0>),
 tensor([[3.2224e-05, 3.5862e-05, 3.8924e-05,  ..., 1.7604e-05, 4.9985e-05,
          4.1200e-05],
         [2.6142e-05, 2.3766e-05, 4.0618e-05,  ..., 1.5519e-05, 5.4291e-05,
          4.2547e-05],
         [9.3734e-05, 2.0246e-05, 2.3997e-05,  ..., 1.6082e-05, 7.1180e-05,
          2.1728e-05],
         ...,
         [3.6948e-05, 2.0667e-05, 2.1793e-05,  ..., 1.9668e-05, 7.0642e-05,
          6.5185e-05],
         [2.8648e-05, 3.0506e-05, 3.9563e-05,  ..., 2.3258e-05, 5.8898e-05,
          8.0002e-05],
         [4.2769e-05, 3.2756e-05, 6.7340e-05,  ..., 1.8599e-05, 6.5449e-05,
          5.3751e-05]], grad_fn=<SoftmaxBackward0>))

In [7]:
#执行训练
def train():
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(5):
        for i, data in enumerate(loader):
            loss, out = model(**data)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            if i % 200 == 0:
                out = out.argmax(dim=1)
                acc = (out == data['label']).sum().item() / len(data['label'])
                print(epoch, i, len(loader), loss.item(), acc)


train()

0 0 1160 9.958359718322754 0.0
0 200 1160 9.958220481872559 0.0
0 400 1160 9.956053733825684 0.125
0 600 1160 9.830792427062988 0.125
0 800 1160 9.841045379638672 0.125


KeyboardInterrupt: 

In [8]:
#执行测试
def test():
    loader_test = torch.utils.data.DataLoader(dataset['test'],
                                              batch_size=8,
                                              shuffle=True,
                                              drop_last=True)

    correct = 0
    total = 0
    for i, data in enumerate(loader_test):
        with torch.no_grad():
            _, out = model(**data)

        out = out.argmax(dim=1)
        correct += (out == data['label']).sum().item()
        total += len(data['label'])

        print(i, len(loader_test), correct / total)

        if i == 5:
            break

    return correct / total


test()

0 144 0.25
1 144 0.1875
2 144 0.16666666666666666
3 144 0.21875
4 144 0.175
5 144 0.1875


0.1875